In [ ]:
import os
import re
import time
from pathlib import Path
import zipfile
import flywheel
from importlib_metadata import files
from collections import defaultdict
from fw_client import FWClient
import io
import tempfile
import os
import shutil
import os
import csv


# ==========================================
# CONFIGURATION
# ==========================================

# Root directory where files will be downloaded.
# Files will be structured as: Root / Project / Subject / Session / Acquisition / filename
LOCAL_SAVE_ROOT = Path("./flywheel_downloads")

# Define exactly what you want to download here.
# Leave lists empty to download ALL items at that level.
DOWNLOAD_CONFIG = {
    'uwophthal': {  # <-- Group ID
        
        'AREDS2_10yr_HRF': {  # <-- Project Name 
            
            # Subject Filter: List specific subject labels, or leave empty [] for ALL
            'subject_filter': [], 
            
            # Session Filter: List specific session labels, or leave empty [] for ALL
            'session_filter': [], 
            
            # Acquisition Filter: Substrings to match in acquisition labels. Empty [] for ALL
            'target_acquisitions': [], 
            
            # File Filter (Optional): Regex pattern to match file names. Use None for ALL
            # Example: r'.*\.zip$' for only zip files, or r'.*\.E2E$' for only E2Es
            'file_regex': None 
        },  
    }
}

# ==========================================
# HELPER FUNCTIONS
# ==========================================

def get_server_api_key() -> str:
    """Reads the Flywheel API key from local api.txt or server_api.txt."""
    for filename in ["api.txt", "server_api.txt"]:
        # key_file = Path(__file__).parent / filename
        key_file = Path().resolve() / filename
        if key_file.exists():
            api_key = key_file.read_text().strip()
            if api_key:
                return api_key
            
    raise FileNotFoundError("API key file not found or is empty. Please create 'api.txt'.")

def sanitize_name(name: str) -> str:
    """Removes illegal characters from strings to safely use them as folder names."""
    if not name:
        return "Unknown"
    return re.sub(r'[^\w\-_\. ]', '_', name).strip()

print("--- Flywheel Batch Downloader Started ---")
start_time = time.time()
total_downloaded = 0

# Connect to Flywheel

api_key = get_server_api_key()
fw = flywheel.Client(api_key)
fw_c = FWClient(api_key=api_key)
print(f"Successfully connected to Flywheel as {fw.get_current_user().id}")

LOCAL_SAVE_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# ==========================================
# MAIN EXECUTION
# ==========================================
import io
import tempfile
import os

print("--- Flywheel Batch Downloader Started ---")
start_time = time.time()
total_downloaded = 0

# Connect to Flywheel

api_key = get_server_api_key()
fw = flywheel.Client(api_key)
fw_c = FWClient(api_key=api_key)
print(f"Successfully connected to Flywheel as {fw.get_current_user().id}")
# except Exception as e:
#     print(f"FATAL: Could not connect to Flywheel: {e}")
    # return

# Ensure local root exists
LOCAL_SAVE_ROOT.mkdir(parents=True, exist_ok=True)

# Loop through Groups
for group_id, projects in DOWNLOAD_CONFIG.items():
    print(f"\nScanning Group: {group_id}")
    try:
        group = fw.get(group_id)
    except Exception as e:
        print(f"  ERROR: Could not access Group {group_id}. Skipping. {e}")
        continue
    
    # Loop through Projects
    for project_name, config in projects.items():
        try:
            project = group.projects.find_first(f'label="{project_name}"')
            if not project:
                raise Exception(f"Project '{project_name}' not found.")
            print(f"  Processing Project: {project.label}")
        except Exception as e:
            print(f"  ERROR: {e}")
            continue
        
    # --- 2. Fetch Existing Tasks & Map by File ID ---
    print("Fetching existing tasks from the project...")
    response = fw_c.get(
        f"/api/readertasks/project/{project.id}", params={"limit": 10000}
    )
    all_tasks = response.get("results", [])

    print(f"Found {len(all_tasks)} total tasks.")
    complete_tasks = [i for i in all_tasks if i.get("status") == "Complete"]
    print(f"Total complete tasks: {len(complete_tasks)}")
    
    job_ids = []
    gear = fw.lookup("gears/mask-exporter") 
    for i in complete_tasks:
        if i["parent_info"]["file"]:
            subject_name = i["parent_info"]["subject"]["label"]
            subject = project.subjects.find_first(f'label="{subject_name}"')
            sessions = subject.sessions()
            for session in sessions:
                for acq in session.acquisitions():
                    for file in acq.files:
                        if file.name == i["parent_info"]["file"]["name"]:
                            inputs = {'input-file': file}
                            destination_container = fw.get(file.parent.id)
                            job_id = gear.run(inputs=inputs, destination=destination_container)
                            job_ids.append(job_id)
                        
        else:
            subject_name = i["parent_info"]["subject"]["label"]
            subject = project.subjects.find_first(f'label="{subject_name}"')
            sessions = subject.sessions()
            for session in sessions:
                for acq in session.acquisitions():
                    job_id = None
                    for file in acq.files:
                        inputs = {'input-file': file}
                        destination_container = fw.get(file.parent.id)
                        job_id = gear.run(inputs=inputs, destination=destination_container)
                        job_ids.append(job_id)

In [ ]:
df = pd.read_csv("job_ids.csv")
job_ids = df["job_id"].tolist()
count = 0
failed_jobs = []
for job_id in job_ids:
    count += 1
    print(f"Processing job {count}/{len(job_ids)}")
    job = fw.get_job(job_id)
    keep_going = False
    while True:
        job = fw.get_job(job_id)
        if job["state"] == "complete":
            keep_going = True
            break
        if job["state"] == "failed":
            failed_jobs.append(job_id)
            break
        time.sleep(1)
    if keep_going:
        if job["outputs"]:
            # print(job["outputs"])
            for file in job["outputs"]:
                if file["name"].endswith(".gz"):
                    file.download('output_downloads/' + file['name'])
                    input_file = job["inputs"]["input-file"]
                    container = fw.get(input_file["id"])
                    container.download_file(input_file["name"], 'input_downloads/' + input_file["name"])

In [ ]:
input_f = job["inputs"]["input-file"]
container = fw.get(input_f["id"])
container.download_file(input_f["name"], 'input_downloads/' + input_f["name"])

In [ ]:
from datetime import datetime, timezone

me = fw.get_current_user()
today_start = datetime.now(timezone.utc).strftime("%Y-%m-%dT00:00:00.000Z")
jobs_today = fw.jobs.find(f'origin.id={me.id},created>{today_start}')
new_job_ids = [j.id for j in jobs_today]
print(new_job_ids)

In [ ]:
import pandas as pd

pd.DataFrame({"new_job_id": new_job_ids}).to_csv("new_job_idss.csv", index=False)

In [ ]:
count = 0
df = pd.read_csv("new_job_idss.csv")
failed_jobs = df["new_job_id"].tolist()
failed_again = []
for job_id in failed_jobs:
    count += 1
    print(f"Processing job {count}/{len(failed_jobs)}")
    job = fw.get_job(job_id)
    keep_going = False
    while True:
        job = fw.get_job(job_id)
        if job["state"] == "complete":
            keep_going = True
            break
        if job["state"] == "failed":
            failed_again.append(job_id)
            break
        time.sleep(1)
    if keep_going:
        if job["outputs"]:
            for file in job["outputs"]:
                if file["name"].endswith(".gz"):
                    file.download('output_downloads/' + file['name'])
                    input_file = job["inputs"]["input-file"]
                    container = fw.get(input_file["id"])
                    container.download_file(input_file["name"], 'input_downloads/' + input_file["name"])

In [ ]:
import shutil
import os
import csv

input_folder = 'testset-inference-ready-full'
output_folder = 'testset-inference-ready-full-renamed'
mapping_file = 'filename_mapping.csv'

find_str = ' + '
replace_str = '_'

os.makedirs(output_folder, exist_ok=True)

mapping = []

for fname in os.listdir(input_folder):
    src_path = os.path.join(input_folder, fname)

    if not os.path.isfile(src_path):
        continue

    new_fname = fname.replace(find_str, replace_str)
    dst_path = os.path.join(output_folder, new_fname)

    shutil.copy2(src_path, dst_path)
    mapping.append({'original_name': fname, 'new_name': new_fname})
    print(f'{fname} -> {new_fname}')

with open(mapping_file, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['original_name', 'new_name'])
    writer.writeheader()
    writer.writerows(mapping)

print(f'Mapping saved to {mapping_file}')

In [ ]:
# For testset ONLY

folder_to_revert = 'testset_predictions_with0s'  # wherever the renamed files currently are
reverted_folder = 'testset_predictions-remapped'       # new folder for reverted files
mapping_file = 'filename_mapping.csv'

os.makedirs(reverted_folder, exist_ok=True)

with open(mapping_file) as f:
    reader = csv.DictReader(f)
    for row in reader:
        current_path = os.path.join(folder_to_revert, row['new_name'])
        original_path = os.path.join(reverted_folder, row['original_name'])

        if os.path.exists(current_path):
            shutil.copy2(current_path, original_path)
            print(f"{row['new_name']} -> {row['original_name']}")
        else:
            print(f"Skipped (not found): {row['new_name']}")

In [ ]:
input_folder = '/home/shared/projects/HRF_Reeva/testset_predictions'
output_folder = '/home/shared/projects/HRF_Reeva/testset_predictions_with0s'

os.makedirs(output_folder, exist_ok=True)

for fname in os.listdir(input_folder):
    src_path = os.path.join(input_folder, fname)

    if not os.path.isfile(src_path):
        continue

    name, ext = os.path.splitext(fname)
    new_fname = f"{name}_0000{ext}"
    dst_path = os.path.join(output_folder, new_fname)

    shutil.copy2(src_path, dst_path)
    print(f"{fname} -> {new_fname}")